In [ ]:
import pandas as pd
from pyhealth.medcode import InnerMap
import pandas as pd

atc = InnerMap.load("ATC")
df_p_code_smoothed = pd.read_csv(f'saved_files/mimic4/mimic4_code_marginal_probs_ccs.csv')
df_ccs_px_labels = pd.read_csv("datasets/CCS_maping/ccs_proc_category_labels_clean.csv",
                               dtype=str, keep_default_na=False, na_filter=False, engine="python")
df_ccs_dx_labels = pd.read_csv("datasets/CCS_maping/ccs_category_labels_clean.csv",
                               dtype=str, keep_default_na=False, na_filter=False, engine="python")
df_p_code_smoothed['org_code']=df_p_code_smoothed['code'].apply(lambda x:x[3:])
df_p_code_smoothed['type']=df_p_code_smoothed['code'].apply(lambda x:x[:2])
dx_name_map = df_ccs_dx_labels.set_index('level_code')['level_label']
px_name_map = df_ccs_px_labels.set_index('level_code')['level_label']

atc = InnerMap.load("ATC")


df_p_code_smoothed3 = pd.read_csv(f'saved_files/mimic3/mimic3_code_marginal_probs_ccs.csv')
df_p_code_smoothed3['org_code']=df_p_code_smoothed['code'].apply(lambda x:x[3:])
df_p_code_smoothed3['type']=df_p_code_smoothed['code'].apply(lambda x:x[:2])

In [ ]:
def get_label_name(row):
    code = row['org_code']
    if row['type']=='rx':
        name = atc.lookup(code)
    elif row['type'] =='dx':
        name = dx_name_map[code]
    elif row['type'] =='px':
        name = px_name_map[code]

    return name

df_p_code_smoothed['label'] = df_p_code_smoothed.apply(get_label_name, axis=1)
df_p_code_smoothed3['label'] = df_p_code_smoothed3.apply(get_label_name, axis=1)

In [ ]:
def build_explain_code_prompt_minimal(
    *,
    code_id: str,
    code_label: str,
    code_type: str,    # {"dx","px","rx"}  (dx=CCS diagnosis, px=CCS procedure, rx=ATC)
    p_code: float,
    ds_name: str,
    target_words: int = 180      # control verbosity; ~150–220 is good for embeddings
) -> str:
    """
    Minimal prompt: get one high-quality descriptive paragraph suitable for embedding.
    Output JSON only:
      {
        "code_id": "...",
        "code_label": "...",
        "code_type": "...",
        "description": "..."       # one dense paragraph
      }
    """

    def fmt_prob(x, nd=6):
        if x is None:
            return "NA"
        try:
            return f"{float(x):.{nd}f}"
        except Exception:
            return "NA"


    p_str = fmt_prob(p_code)

    ds_context = (
        f"{ds_name} => ICU+inpatient EHR; visit-level codes; adult population; "
        "visit = one hospitalization (admit→discharge)"
        if ds_name else
        "ICU+inpatient EHR; visit-level codes; adult population; visit = hospitalization"
    )
    type_hint = {
        "dx": "CCS diagnosis",
        "px": "CCS procedure",
        "rx": "ATC drug"
    }.get(code_type, code_type)

    # very short, type-aware nudge to shape the paragraph
    # guidance = {
    #     "dx": "Define the condition, typical clinical picture, key risks/complications, and usual high-level management patterns.",
    #     "px": "State purpose, indications/contraindications, typical steps in care episodes, and key risks/monitoring.",
    #     "rx": "State class/mechanism (plain language), core indications, major interactions/contraindications, and high-signal adverse effects."
    # }.get(code_type, "Provide a clinically useful explanation suitable for clinicians.")
    # Rich, type-aware guidance: push for portable, high-signal clinical facts

    guidance_dx = (
        "Define the condition succinctly; typical clinical presentation and key red flags; common etiologies and "
        "high-level pathophysiology; major risk factors and prevalent comorbidities; core diagnostic approach "
        "(history/exam cues and hallmark labs/imaging—no numeric cutoffs); severity/staging concepts; brief management "
        "overview (first-line modalities, supportive care, when to escalate/consult); complications to watch for and "
        "longitudinal outcomes; closely related or easily confused diagnoses and one cue to distinguish them. "
        "Keep to class-level statements."
    )
    guidance_px = (
        "State the procedure’s purpose and primary indications; prerequisites/contraindications and perioperative prep; "
        "a high-level sense of how it is performed (steps/components, anesthesia—no device brands); immediate and delayed "
        "risks/complications and how they are mitigated; typical monitoring/aftercare; common alternatives or non-procedural "
        "options and when preferred; how this fits in the care pathway. Keep to class-level statements."
    )
    guidance_rx = (
        "Identify the pharmacologic class and plain-language mechanism; core indications and common clinically meaningful "
        "off-label uses (no marketing claims); major contraindications and black-box concerns; high-signal interaction "
        "patterns (drug–drug, drug–condition); adverse effects emphasizing serious/frequent events and monitoring; practical "
        "clinical pearls (onset, adherence, organ-adjustment considerations) without doses or lab thresholds; briefly relate "
        "to nearby classes. Keep to class-level statements; do not list specific products."
    )
    guidance = {"dx": guidance_dx, "px": guidance_px, "rx": guidance_rx}.get(
        code_type, "Provide portable, high-signal clinical context useful for clinicians."
    )




    return f"""
You are a medical reasoning assistant. Return **JSON only**. No markdown, no extra fields.
Write **one dense paragraph (~{target_words} words)** giving most important/accurate clinical context about a medical concept (code) for embedding.
Use established medical knowledge.

### Code
- id: {code_id}
- label: {code_label}
- type: {type_hint}
- marginal P(code) in dataset: {p_str}
- dataset context: {ds_context}

### Focus (ideal information to provide, if exists)
{guidance}


### Constraints
- Write one cohesive paragraph (~{target_words} words), declarative and compact. Be accurate and concise; prefer well-established clinical knowledge.
- Widely accepted, class-level clinical facts appropriate to the label/type. If label is broad/ambiguous, provide a generic overview and state the ambiguity; do not guess
- Avoid numeric dosing, exact thresholds, or hospital-specific policies.
- Do **not** invent local coding rules, dosages, or precise rates.
- NO lists, headings, bullets, numbered steps, citations, or URLs. NO markdown—return pure JSON.

### Forbidden content
- Specific product/brand names, device SKUs, URLs, dosing, numeric lab thresholds, hospital policies, billing rules.
- ATC/CCS subdivisions not supplied in the prompt.
- Speculative claims, local statistics, or unverified niche practices.

### Output (JSON only)
{{
  "code_id": "{code_id}",
  "code_label": "{code_label}",
  "code_type": "{code_type}",
  "description": "<one dense paragraph ~{target_words} words; no lists/URLs/doses/thresholds>"
}}
""".strip()





In [ ]:
import os, json, time, hashlib
import pandas as pd
from typing import Optional, List, Dict
from openai import OpenAI, APIStatusError

# --- config ---
MODEL_PRIMARY  = "gpt-5"     # your GPT-5 family model; adjust to the exact name you have
MODEL_FALLBACK = "gpt-5-mini"      # safe fallback
OUT_PATH       = "node_text_embeddings_input.parquet"  # or .csv
ERRORS_JSONL   = "node_text_errors.jsonl"
DS_NAME        = "MIMIC-III/IV"  # short dataset label for context in the prompt

# --- client ---
api_key = "s--"
client = OpenAI(api_key=api_key)

# --- tiny utils ---
def prompt_hash(prompt: str) -> str:
    return hashlib.sha1(prompt.encode("utf-8")).hexdigest()

def first_json_block(text: str) -> str:
    start = text.find("{")
    if start == -1:
        raise json.JSONDecodeError("No JSON object start found", text, 0)
    depth = 0
    for i, ch in enumerate(text[start:], start=start):
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    raise json.JSONDecodeError("No complete JSON object found", text, start)



def _call_openai_once(prompt: str, *, model: str, temperature: float = 0.2) -> dict:
    """
    Single GPT-5 call, returns strict JSON. Raises on API or JSON errors.
    """
    resp = client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}],
        text={
            "format": {
                "type": "json_schema",
                "name": "code_metadata",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "code_id": {"type": "string"},
                        "code_label": {"type": "string"},
                        "code_type": {"type": "string"},
                        "description": {"type": "string"},
                    },
                    "required": ["code_id", "code_label", "code_type", "description"],
                    "additionalProperties": False
                }
            }
        },
    )
    txt = resp.output_text
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return json.loads(first_json_block(txt))



class BillingError(Exception): pass


def openai_json_with_retries(
    prompt: str,
    *,
    attempts: int = 3,
    sleep: float = 1.5,
    primary: str = MODEL_PRIMARY,
    fallback: str = MODEL_FALLBACK,
    temperature: float = 0.2,
) -> dict:
    """
    Try primary twice, then fallback; exponential backoff.
    Raises BillingError for quota/billing; RuntimeError after final failure.
    """
    models = [primary, primary, fallback]
    last_err = None
    for i in range(attempts):
        model = models[min(i, len(models)-1)]
        try:
            return _call_openai_once(
                prompt,
                model=model,
                temperature=temperature,
            )
        except APIStatusError as e:
            status = getattr(e, "status_code", None)
            msg = str(e)
            if "insufficient_quota" in msg or "billing_hard_limit" in msg or status in (401,402):
                raise BillingError("OpenAI quota/billing/WrongKey Error.") from e
            if status in (408, 409, 429, 500, 502, 503, 504):
                last_err = e
            else:
                raise
        except (json.JSONDecodeError, Exception) as e:
            last_err = e
        time.sleep(sleep * (2 ** i))
    raise RuntimeError(f"LLM call failed after {attempts} attempts: {last_err}")


def load_out(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        return pd.DataFrame(columns=[
            "code","org_code","type","label","P(code)",
            "desc_text","model_id","prompt_id","raw_json"
        ])
    ext = os.path.splitext(path)[1].lower()
    return pd.read_parquet(path) if ext in {".parquet",".pq"} else pd.read_csv(path)

def flush_out(df: pd.DataFrame, path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    ext = os.path.splitext(path)[1].lower()
    if ext in {".parquet",".pq"}:
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)

def write_error_jsonl(path: str, obj: dict):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

# --- your prompt builder (assumed available in scope) ---
# from your_module import build_explain_code_prompt_minimal


from typing import List, Dict
from tqdm.auto import tqdm

def flush_now(out_df: pd.DataFrame, batch_rows: List[Dict], out_path: str) -> pd.DataFrame:
    """Atomically append any pending rows and write to disk."""
    if batch_rows:
        out_df = pd.concat([out_df, pd.DataFrame(batch_rows)], ignore_index=True)
        flush_out(out_df, out_path)
        batch_rows.clear()  # important: empty in-place so the caller's list is reset
    return out_df

def generate_code_descriptions(
    df: pd.DataFrame,
    *,
    out_path: str = OUT_PATH,
    errors_path: str = ERRORS_JSONL,
    ds_name: str = DS_NAME,
    batch_size: int = 100
):
    out_df = load_out(out_path)
    seen = set(out_df["code"]) if len(out_df) else set()

    # Only process new rows
    todo_df = df[~df["code"].isin(seen)].copy()

    batch_rows: List[Dict] = []

    try:
        for _, row in tqdm(
            todo_df.iterrows(),
            total=len(todo_df),
            desc="Generating code descriptions",
            unit="code",
            miniters=1,
        ):
            try:
                code     = row["code"]
                p_code   = row["P(code)"]
                org_code = row["org_code"]
                ctype    = row["type"]      # "dx" | "px" | "rx"
                label    = row["label"]
            except KeyError as e:
                write_error_jsonl(errors_path, {"code": row.get("code", None), "err": f"missing_column:{e}"})
                continue

            if code in seen:
                continue

            prompt = build_explain_code_prompt_minimal(
                code_id=org_code,
                code_label=label,
                code_type=ctype,
                p_code=p_code,
                ds_name=ds_name,
                target_words=220,
            )
            p_id = prompt_hash(prompt)

            # LLM call
            try:
                obj = openai_json_with_retries(prompt, attempts=3)
            except BillingError:
                write_error_jsonl(errors_path, {"code": code, "org_code": org_code, "err": "billing_quota"})
                # Hard stop: flush what we have so far, then re-raise to bubble up
                out_df = flush_now(out_df, batch_rows, out_path)
                raise
            except Exception as e:
                write_error_jsonl(errors_path, {"code": code, "org_code": org_code, "err": f"llm:{e}"})
                # Soft failure: flush immediately so we never lose progress, then continue
                out_df = flush_now(out_df, batch_rows, out_path)
                break

            # Validate minimal JSON schema
            try:
                desc = (obj.get("description") or "").strip()
                if not desc:
                    raise ValueError("empty description")
            except Exception as e:
                write_error_jsonl(errors_path, {"code": code, "org_code": org_code, "err": f"validate:{e}", "raw": obj})
                # Flush on validation error too (keeps progress on long runs)
                out_df = flush_now(out_df, batch_rows, out_path)
                break

            # Accumulate one clean row
            batch_rows.append({
                "code": code,
                "org_code": org_code,
                "type": ctype,
                "label": label,
                "P(code)": p_code,
                "desc_text": desc,
                "model_id": MODEL_PRIMARY,
                "prompt_id": p_id,
                "raw_json": json.dumps(obj, ensure_ascii=False),
            })
            seen.add(code)

            # Periodic flush
            if len(batch_rows) >= batch_size:
                out_df = flush_now(out_df, batch_rows, out_path)

    except KeyboardInterrupt:
        # Flush immediately on Ctrl+C, then re-raise so caller knows it stopped
        out_df = flush_now(out_df, batch_rows, out_path)
        raise
    except Exception as e:
        # Any other unexpected exception: flush and re-raise
        out_df = flush_now(out_df, batch_rows, out_path)
        raise
    finally:
        # Final guaranteed flush (covers normal completion too)
        out_df = flush_now(out_df, batch_rows, out_path)




In [ ]:
# --- config ---
MODEL_PRIMARY  = "gpt-5"     # your GPT-5 family model; adjust to the exact name you have
MODEL_FALLBACK = "gpt-5-mini"      # safe fallback
OUT_PATH       = "saved_files/mimic4/KG_openai/node_text_embeddings_input.parquet"  # or .csv
ERRORS_JSONL   = "saved_files/mimic4/KG_openai/node_text_errors.jsonl"
DS_NAME        = "MIMIC-III/IV"  # short dataset label for context in the prompt

generate_code_descriptions(df=df_p_code_smoothed_new, out_path=OUT_PATH, errors_path=ERRORS_JSONL,ds_name=DS_NAME,batch_size=50)